# HCDE: train, evaluate and export the assistance-calibrated model

**Before you run:** Settings → Accelerator **GPU T4 x1**, Internet **on**.
Add-ons → Secrets (only the ones you need):

| Secret | When |
| --- | --- |
| `GITHUB_TOKEN` | only if the AnchorAI repo is private |
| `TEACHER_BASE_URL`, `TEACHER_API_KEY`, `TEACHER_MODEL` | for the judge + disclosure scores in step 4 |
| `JUDGE_MODEL` | optional: judge with a different model from the teacher |

Run the cells in order. Training takes about 1-2 hours for the 7B model on one T4.

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
def secret(name):
    try:
        return secrets.get_secret(name)
    except Exception:
        return ""

BRANCH = "dev/dulanja"          # the branch that holds train/val/test.jsonl
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"   # fallback if out of memory: Qwen/Qwen2.5-3B-Instruct
RUN = "/kaggle/working/runs/hcde-qwen7b"

token = secret("GITHUB_TOKEN")
url = f"https://{token}@github.com/DulanjaR/AnchorAI.git" if token else "https://github.com/DulanjaR/AnchorAI.git"
if not os.path.exists("/kaggle/working/AnchorAI"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, "/kaggle/working/AnchorAI"], check=True)
os.chdir("/kaggle/working/AnchorAI")
for k in ["TEACHER_BASE_URL", "TEACHER_API_KEY", "TEACHER_MODEL", "TEACHER_EXTRA",
          "JUDGE_MODEL", "JUDGE_EXTRA", "JUDGE_MAX_TOKENS"]:
    if secret(k):
        os.environ[k] = secret(k)
!git log --oneline -1
!ls -la components/c4_hcde/training/data/dataset/

In [ ]:
!pip install -q -U peft bitsandbytes accelerate
!nvidia-smi --query-gpu=name,memory.total --format=csv
import transformers, peft, torch; print(transformers.__version__, peft.__version__, torch.__version__)

## 1. Train (QLoRA)

In [ ]:
!python -m components.c4_hcde.training.train_qlora --base-model {BASE_MODEL} --out {RUN}

## 2. Quick look: one task at all five tiers

In [ ]:
import json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from components.c4_hcde.training.tiers import MODES, control_system_prompt

tok = AutoTokenizer.from_pretrained(f"{RUN}/adapter")
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
model = PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map={"": 0}), f"{RUN}/adapter")
task = json.loads(open("components/c4_hcde/training/data/dataset/test.jsonl").readline())["messages"][1]["content"]
print("TASK:", task, "\n")
for mode in MODES:
    msgs = [{"role": "system", "content": control_system_prompt(mode)}, {"role": "user", "content": task}]
    text = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    enc = tok(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    out = model.generate(**enc, max_new_tokens=400, do_sample=False)
    print(f"===== {mode} =====\n{tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)}\n")
del model; torch.cuda.empty_cache()

## 3. Evaluate on held-out IT tasks (3 conditions)
Add `--judge` once the TEACHER secrets are set (adds disclosure + judge scores).

In [ ]:
judge = "--judge" if os.environ.get("TEACHER_BASE_URL") else ""
!python -m components.c4_hcde.training.evaluate --base-model {BASE_MODEL} --adapter {RUN}/adapter --out {RUN}/eval {judge}

## 4. Transfer test: 20 non-IT tasks the model never saw

In [ ]:
!python -m components.c4_hcde.training.evaluate --base-model {BASE_MODEL} --adapter {RUN}/adapter \
    --test components/c4_hcde/training/data/dataset/transfer_raw.jsonl \
    --elements components/c4_hcde/training/data/dataset/transfer_elements.json \
    --out {RUN}/eval-transfer {judge}

## 5. Export to GGUF for Ollama on your Mac
Restart the kernel first if memory is low, then re-run the first cell and this one.

In [ ]:
!apt-get -qq install -y cmake > /dev/null
!python -m components.c4_hcde.training.export_gguf --base-model {BASE_MODEL} --adapter {RUN}/adapter --out /kaggle/working/gguf

## 6. Package results for download
Download `hcde-results.zip` and the two files in `/kaggle/working/gguf` from the Output panel.

In [ ]:
!cd /kaggle/working && zip -qr hcde-results.zip runs/hcde-qwen7b/adapter runs/hcde-qwen7b/training_meta.json runs/hcde-qwen7b/eval runs/hcde-qwen7b/eval-transfer
!ls -lh /kaggle/working /kaggle/working/gguf